# Spotify Lyrics Modeling 
_(lyrics-only expert)_

### What this notebook does
- Loads Spotify rows that still contain identifiers (`spotify_tracks_viral.parquet`).
- Loads cleaned lyrics (`lyrics_cleaned.csv`).
- Builds an **ephemeral alignment** using normalized `(artist_norm, title_norm)` keys.
- Trains a lyrics-only classifier **only on rows where lyrics exist**.
- Writes `p_b` predictions for **all rows** (NaN where no lyrics), for downstream ensembling.

### Important pipeline note
- `04_process_lyrics.ipynb` saves a fitted TF‑IDF vectorizer to `notebooks/models/lyrics_tfidf.joblib`.
- This notebook **loads that vectorizer** so Model B uses the **same vocabulary** as preprocessing.

### Output
- `data/processed/preds_model_b.parquet` — `p_b` from **calibrated** Model B (local)
- `notebooks/models/model_b_*.joblib` + `notebooks/models/metadata/model_b_metadata.json` — baseline, random-search & grid-search clfs, **selected** tuned clf, calibrated estimator

### What to run first
1. `04_clean_lyrics.ipynb` → `data/cleaned/lyrics_cleaned.csv`
2. `04_process_lyrics.ipynb` → `notebooks/models/lyrics_tfidf.joblib`
3. This notebook (`06`) through the end — generates saved models + `preds_model_b.parquet` for `07_meta_combiner.ipynb`


### 1. Paths & sanity checks

In [1]:
import os
import re
import unicodedata
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import json
from datetime import datetime, timezone

from scipy.stats import loguniform
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split

IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    SPOTIFY_VIRAL = Path("/kaggle/working/spotify_tracks_viral.parquet")
    LYRICS_CLEAN = Path("/kaggle/working/lyrics_cleaned.csv")
    LYRICS_VEC = Path("/kaggle/working/notebooks/models/lyrics_tfidf.joblib")
    MODELS_DIR = Path("/kaggle/working/notebooks/models")
    OUT_PREDS = Path("/kaggle/working/preds_model_b.parquet")
else:
    SPOTIFY_VIRAL = Path("../data/processed/spotify_tracks_viral.parquet")
    LYRICS_CLEAN = Path("../data/cleaned/lyrics_cleaned.csv")
    LYRICS_VEC = Path("../notebooks/models/lyrics_tfidf.joblib")
    MODELS_DIR = Path("../notebooks/models")
    OUT_PREDS = Path("../data/processed/preds_model_b.parquet")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
META_DIR = MODELS_DIR / "metadata"
META_DIR.mkdir(parents=True, exist_ok=True)
OUT_CLF_BASELINE = MODELS_DIR / "model_b_clf_baseline.joblib"
OUT_CLF_RANDOM = MODELS_DIR / "model_b_clf_randomsearch.joblib"
OUT_CLF_GRID = MODELS_DIR / "model_b_clf_gridsearch.joblib"
OUT_CLF_TUNED = MODELS_DIR / "model_b_clf_tuned.joblib"
OUT_CAL = MODELS_DIR / "model_b_calibrated.joblib"
OUT_META = META_DIR / "model_b_metadata.json"

print("IS_KAGGLE:", IS_KAGGLE)
print("SPOTIFY_VIRAL exists:", SPOTIFY_VIRAL.exists(), SPOTIFY_VIRAL)
print("LYRICS_CLEAN exists:", LYRICS_CLEAN.exists(), LYRICS_CLEAN)
print("LYRICS_VEC exists:", LYRICS_VEC.exists(), LYRICS_VEC)
print("MODELS_DIR:", MODELS_DIR)
print("META_DIR:", META_DIR)
print("OUT_PREDS:", OUT_PREDS)


IS_KAGGLE: False
SPOTIFY_VIRAL exists: False ..\data\processed\spotify_tracks_viral.parquet
LYRICS_CLEAN exists: False ..\data\cleaned\lyrics_cleaned.csv
LYRICS_VEC exists: False ..\notebooks\models\lyrics_tfidf.joblib
MODELS_DIR: ..\notebooks\models
META_DIR: ..\notebooks\models\metadata
OUT_PREDS: ..\data\processed\preds_model_b.parquet


### 2. Load data

- sp = `spotify_tracks_viral.parquet`
- lyr = `lyrics_cleaned.csv`
- vec = `lyrics_tfidf.joblib`

In [20]:
sp = pd.read_parquet(SPOTIFY_VIRAL)
lyr = pd.read_csv(LYRICS_CLEAN)

vec = joblib.load(LYRICS_VEC)

sp.shape, lyr.shape, type(vec)

((113999, 22), (57438, 8), sklearn.feature_extraction.text.TfidfVectorizer)

### 3. Normalize keys for alignment

In [21]:
_RE_FEAT = re.compile(r"\s*\b(?:feat\.?|ft\.?|featuring)\b.*$", flags=re.IGNORECASE)
_RE_PARENS = re.compile(r"\s*[\(\[].*?[\)\]]\s*")
_RE_NON_ALNUM = re.compile(r"[^a-z0-9\s]+")


def _strip_accents(s: str) -> str:
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch)
    )


def norm_artist_primary(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    if not s:
        return ""
    # Spotify often uses ';' for multi-artist strings; use the first credited artist.
    s = s.split(";", 1)[0].strip()
    s = _RE_FEAT.sub("", s)
    s = _strip_accents(s)
    s = _RE_NON_ALNUM.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def norm_title(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    if not s:
        return ""
    s = _strip_accents(s)
    s = _RE_PARENS.sub(" ", s)
    if " - " in s:
        s = s.split(" - ", 1)[0]
    s = _RE_FEAT.sub("", s)
    s = s.replace("&", " and ")
    s = _RE_NON_ALNUM.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


### 4. Ephemeral merge (no persisted join table)

In [22]:
sp = sp.copy()
sp["row_id"] = np.arange(len(sp), dtype=np.int64)

sp["artist_norm"] = sp["artists"].map(norm_artist_primary)
sp["title_norm"] = sp["track_name"].map(norm_title)

m = sp.merge(
    lyr[["artist_norm", "title_norm", "lyrics"]],
    on=["artist_norm", "title_norm"],
    how="left",
    validate="m:1",
)

m["has_lyrics"] = m["lyrics"].notna() & (m["lyrics"].astype(str) != "")

m["has_lyrics"].mean(), int(m["has_lyrics"].sum())


(np.float64(0.040728427442345984), 4643)

Only **4.1%** of Spotify tracks got a successful lyrics match for the merge key that we used.   

And only 4640 tracks have non-empty lyrics after the join but we still keep all rows from the left join (Spotify Tracks).

### 5. Train/val/test split (by `track_id`)

In [23]:
keys = m[["track_id", "viral"]].drop_duplicates("track_id")

# Create ~70% train / 10% val / 20% test at the track_id level (stratified).
# Step 1: hold out 30% as (val + test).
train_ids, temp_ids = train_test_split(
    keys["track_id"],
    test_size=0.3,
    random_state=42,
    stratify=keys["viral"],
)

# Step 2: split the 30% temp into 10% val and 20% test => val is 1/3 of temp.
val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=(2 / 3),
    random_state=42,
    stratify=keys.set_index("track_id").loc[temp_ids, "viral"],
)

id_to_split = {}
for tid in train_ids:
    id_to_split[tid] = "train"
for tid in val_ids:
    id_to_split[tid] = "val"
for tid in test_ids:
    id_to_split[tid] = "test"

m["split"] = m["track_id"].map(id_to_split)
m["split"].value_counts(dropna=False)


split
train    79732
test     22852
val      11415
Name: count, dtype: int64

### 6. Model B: baseline → tune (random vs grid) → pick best → calibrate → save

- **Frozen TF‑IDF** from `lyrics_tfidf.joblib`; features are `vec.transform(lyrics)`.
- **Baseline**: default `LogisticRegression` for comparison.
- **Tuning**: `RandomizedSearchCV` and `GridSearchCV` on the **train** subset (matched lyrics only), same `CV_FOLDS` and scoring = average precision; we compare CV scores and **calibrate the winner**.
- **Calibration**: `CalibratedClassifierCV` on the selected estimator.
- Artifacts: `model_b_clf_*.joblib`, `model_b_calibrated.joblib`, `metadata/model_b_metadata.json`.


In [24]:
def eval_split(name: str, Xp, yp, model) -> dict:
    if len(yp) == 0:
        print(name, "n=0 (skip)")
        return {"n": 0, "roc_auc": None, "pr_auc": None}
    proba = model.predict_proba(Xp)[:, 1]
    out = {
        "n": int(len(yp)),
        "roc_auc": float(roc_auc_score(yp, proba)),
        "pr_auc": float(average_precision_score(yp, proba)),
    }
    print(name, "n=", out["n"], "ROC-AUC=", out["roc_auc"], "PR-AUC=", out["pr_auc"])
    return out


train_mask = (m["split"] == "train") & m["has_lyrics"]
val_mask = (m["split"] == "val") & m["has_lyrics"]
test_mask = (m["split"] == "test") & m["has_lyrics"]

X_train = vec.transform(m.loc[train_mask, "lyrics"].fillna(""))
y_train = m.loc[train_mask, "viral"].astype(int).to_numpy()
X_val = vec.transform(m.loc[val_mask, "lyrics"].fillna(""))
y_val = m.loc[val_mask, "viral"].astype(int).to_numpy()
X_test = vec.transform(m.loc[test_mask, "lyrics"].fillna(""))
y_test = m.loc[test_mask, "viral"].astype(int).to_numpy()

X_train.shape, y_train.shape, X_val.shape, X_test.shape


((3235, 128303), (3235,), (480, 128303), (928, 128303))

**ROC-AUC (Receiver Operating Characteristic – Area Under Curve)**

Range: 0 to 1 (often reported as 0.0–1.0).

What it measures: How well predicted probabilities order rows: higher scores for true positives (viral=1) than for true negatives (viral=0), across all possible thresholds.

How to interpret:
- 0.5 ≈ random ordering (no discriminative power).
- 1.0 = perfect separation (in practice rare).
- 0.7 is often “decent” for noisy real data; > 0.8 is strong — but it depends on difficulty and leakage.

Caveat: With strong class imbalance, ROC-AUC can still look optimistic if the model is good at ranking negatives; it doesn’t fully tell you how good you are at finding the rare positive class.

**PR-AUC (Precision–Recall AUC / Average Precision)**

Range: 0 to 1.

What it measures: Quality of positive class prediction across thresholds — combines precision and recall in one number. Very relevant when viral=1 is rare.

How to interpret:
- Compare to baseline prevalence: if only ~24% of rows are viral, a dumb model that always predicts “viral” has precision ≈ 0.24 on positives; PR-AUC near the positive rate can mean weak signal.
- Higher is better; gains over a simple baseline (e.g. always predict majority class, or random) matter more than the raw number alone.

Accuracy = fraction of correct hard predictions (0/1) at threshold 0.5.

ROC-AUC / PR-AUC use full probability rankings (or all thresholds), so they answer: 

- “Does the model rank viral tracks higher?” not “How many labels did we get exactly right at 50%?”

**Practical Reading**

Use ROC-AUC for a global sense of ranking quality.

Use PR-AUC when you care about catching viral tracks under imbalance.

Always note: these are computed only on lyrics-matched rows — they describe Model B on covered data, not necessarily how it behaves on the full 114k Spotify rows (where p_b is missing and the meta-model uses Model A).

### 6a: Baseline Classifier (LogisticRegression)

In [25]:
# --- Baseline classifier (reference) ---
clf_baseline = LogisticRegression(max_iter=5000, solver="lbfgs")
clf_baseline.fit(X_train, y_train)
metrics_baseline = {
    "val": eval_split("baseline val", X_val, y_val, clf_baseline),
    "test": eval_split("baseline test", X_test, y_test, clf_baseline),
}
joblib.dump(clf_baseline, OUT_CLF_BASELINE)
print("Saved", OUT_CLF_BASELINE)


baseline val n= 480 ROC-AUC= 0.7585387135835885 PR-AUC= 0.6122665159592291
baseline test n= 928 ROC-AUC= 0.7187346910210652 PR-AUC= 0.6480081299640567
Saved ../notebooks/models/model_b_clf_baseline.joblib


In [26]:
# --- Shared CV settings (increased folds; stratified for classification) ---
CV_FOLDS = 10
LR_BASE = LogisticRegression(max_iter=8000, solver="lbfgs")

### 6b: RandomizedSearchCV

In [27]:
# --- RandomizedSearchCV ---
param_dist = {
    "C": loguniform(1e-3, 1e2),
    "class_weight": [None, "balanced"],
}
search = RandomizedSearchCV(
    LR_BASE,
    param_distributions=param_dist,
    n_iter=30,
    cv=CV_FOLDS,
    scoring="average_precision",
    random_state=42,
    n_jobs=-1,
    refit=True,
    verbose=2,
)
search.fit(X_train, y_train)
clf_random = search.best_estimator_
metrics_random = {
    "val": eval_split("randomsearch val", X_val, y_val, clf_random),
    "test": eval_split("randomsearch test", X_test, y_test, clf_random),
}
joblib.dump(clf_random, OUT_CLF_RANDOM)
print("[RandomizedSearchCV] best_params", search.best_params_)
print("[RandomizedSearchCV] best_cv_average_precision", search.best_score_)
print("Saved", OUT_CLF_RANDOM)


Fitting 10 folds for each of 30 candidates, totalling 300 fits
[CV] END ............C=0.0745934328572655, class_weight=None; total time=   0.3s
[CV] END ............C=0.0745934328572655, class_weight=None; total time=   0.3s
[CV] END ............C=0.0745934328572655, class_weight=None; total time=   0.3s
[CV] END ............C=0.0745934328572655, class_weight=None; total time=   0.3s
[CV] END ............C=0.0745934328572655, class_weight=None; total time=   0.3s
[CV] END ............C=0.0745934328572655, class_weight=None; total time=   0.3s
[CV] END ............C=0.0745934328572655, class_weight=None; total time=   0.3s
[CV] END ............C=0.0745934328572655, class_weight=None; total time=   0.4s
[CV] END ......C=0.008263688714158009, class_weight=balanced; total time=   0.1s
[CV] END ......C=0.008263688714158009, class_weight=balanced; total time=   0.1s
[CV] END ......C=0.008263688714158009, class_weight=balanced; total time=   0.1s
[CV] END ............C=0.0745934328572655, cla

### 6b. GridSearchCV

Discrete grid over `C` and `class_weight` (same `CV_FOLDS` and `average_precision` scoring as RandomizedSearch).

In [28]:
# --- GridSearchCV (same folds / scoring as RandomizedSearch) ---
param_grid = {
    "C": [1e-3, 1e-2, 1e-1, 1.0, 10.0, 1e2],
    "class_weight": [None, "balanced"],
}
grid_search = GridSearchCV(
    LogisticRegression(max_iter=8000, solver="lbfgs"),
    param_grid=param_grid,
    cv=CV_FOLDS,
    scoring="average_precision",
    n_jobs=-1,
    refit=True,
    verbose=2,
)
grid_search.fit(X_train, y_train)
clf_grid = grid_search.best_estimator_
metrics_grid = {
    "val": eval_split("gridsearch val", X_val, y_val, clf_grid),
    "test": eval_split("gridsearch test", X_test, y_test, clf_grid),
}
joblib.dump(clf_grid, OUT_CLF_GRID)
print("[GridSearchCV] best_params", grid_search.best_params_)
print("[GridSearchCV] best_cv_average_precision", grid_search.best_score_)
print("Saved", OUT_CLF_GRID)

Fitting 10 folds for each of 12 candidates, totalling 120 fits
[CV] END .........................C=0.001, class_weight=None; total time=   0.1s
[CV] END .........................C=0.001, class_weight=None; total time=   0.2s
[CV] END .........................C=0.001, class_weight=None; total time=   0.2s
[CV] END .........................C=0.001, class_weight=None; total time=   0.2s
[CV] END .........................C=0.001, class_weight=None; total time=   0.1s
[CV] END .........................C=0.001, class_weight=None; total time=   0.2s
[CV] END .........................C=0.001, class_weight=None; total time=   0.2s
[CV] END .........................C=0.001, class_weight=None; total time=   0.2s
[CV] END .........................C=0.001, class_weight=None; total time=   0.2s
[CV] END .....................C=0.001, class_weight=balanced; total time=   0.1s
[CV] END .....................C=0.001, class_weight=balanced; total time=   0.1s
[CV] END .........................C=0.001, cla

### 6c. Compare CV scores and select estimator for calibration

Pick the search with **higher** mean CV average precision (`best_score_`). On a tie, prefer **grid** (small discrete set).

In [29]:
score_random = float(search.best_score_)
score_grid = float(grid_search.best_score_)
if score_grid >= score_random:
    clf_tuned = clf_grid
    tune_source = "grid"
    print(
        f"Selected GridSearchCV (cv_ap {score_grid:.6f} >= random {score_random:.6f})"
    )
else:
    clf_tuned = clf_random
    tune_source = "randomized"
    print(
        f"Selected RandomizedSearchCV (cv_ap {score_random:.6f} > grid {score_grid:.6f})"
    )

metrics_tuned = {
    "val": eval_split("selected val", X_val, y_val, clf_tuned),
    "test": eval_split("selected test", X_test, y_test, clf_tuned),
}
joblib.dump(clf_tuned, OUT_CLF_TUNED)
print("Saved primary tuned clf →", OUT_CLF_TUNED)

Selected RandomizedSearchCV (cv_ap 0.785467 > grid 0.785363)
selected val n= 480 ROC-AUC= 0.7607824631384003 PR-AUC= 0.6421448390687488
selected test n= 928 ROC-AUC= 0.7112650756992559 PR-AUC= 0.6317795583612413
Saved primary tuned clf → ../notebooks/models/model_b_clf_tuned.joblib


In [30]:
# --- Probability calibration (for stacking on predict_proba) ---
# Sigmoid (Platt) + inner CV; use isotonic only if enough samples per fold.
cal = CalibratedClassifierCV(clf_tuned, method="sigmoid", cv=3)
cal.fit(X_train, y_train)
metrics_cal = {
    "val": eval_split("calibrated val", X_val, y_val, cal),
    "test": eval_split("calibrated test", X_test, y_test, cal),
}
joblib.dump(cal, OUT_CAL)
print("Saved", OUT_CAL)


calibrated val n= 480 ROC-AUC= 0.7556182776551037 PR-AUC= 0.6295150224738695
calibrated test n= 928 ROC-AUC= 0.7131406443184735 PR-AUC= 0.6292978306938717
Saved ../notebooks/models/model_b_calibrated.joblib


In [31]:
def _json_safe(obj):
    if isinstance(obj, dict):
        return {k: _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


meta = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "cv_folds": int(CV_FOLDS),
    "selection": {
        "source": tune_source,
        "randomized_search_cv_ap": score_random,
        "grid_search_cv_ap": score_grid,
        "rule": "higher mean CV average_precision; tie → grid",
    },
    "best_params": _json_safe(
        grid_search.best_params_ if tune_source == "grid" else search.best_params_
    ),
    "best_cv_average_precision": float(
        score_grid if tune_source == "grid" else score_random
    ),
    "randomized_search": {
        "n_iter": 30,
        "cv": int(CV_FOLDS),
        "scoring": "average_precision",
        "best_params": _json_safe(search.best_params_),
        "best_cv_average_precision": float(search.best_score_),
    },
    "grid_search": {
        "param_grid": _json_safe(param_grid),
        "cv": int(CV_FOLDS),
        "scoring": "average_precision",
        "best_params": _json_safe(grid_search.best_params_),
        "best_cv_average_precision": float(grid_search.best_score_),
    },
    "calibration": {"method": "sigmoid", "cv": 3},
    "metrics": {
        "baseline": metrics_baseline,
        "randomized_search": metrics_random,
        "grid_search": metrics_grid,
        "selected_tuned": metrics_tuned,
        "calibrated": metrics_cal,
    },
}
with open(OUT_META, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)
print("Wrote", OUT_META)

# Use calibrated model for downstream p_b
model_b_final = cal


Wrote ../notebooks/models/metadata/model_b_metadata.json


### 7. Emit predictions for ensembling

In [32]:
m["p_b"] = np.nan
mask_pred = m["has_lyrics"]

if mask_pred.any():
    X_all = vec.transform(m.loc[mask_pred, "lyrics"].fillna(""))
    m.loc[mask_pred, "p_b"] = model_b_final.predict_proba(X_all)[:, 1]

preds_b = m[["row_id", "track_id", "has_lyrics", "p_b", "split"]].copy()

OUT_PREDS.parent.mkdir(parents=True, exist_ok=True)
preds_b.to_parquet(OUT_PREDS, index=False)

print("Vectorizer Probabilities saved to:", OUT_PREDS)
OUT_PREDS, preds_b.head()

Vectorizer Probabilities saved to: ../data/processed/preds_model_b.parquet


(PosixPath('../data/processed/preds_model_b.parquet'),
    row_id                track_id  has_lyrics  p_b  split
 0       0  5SuOikwiRyPMVoIQDJUgSV       False  NaN  train
 1       1  4qPNDBW1i3p13qLCt0Ki3A       False  NaN    val
 2       2  1iJBSr7s7jYXzM8EGcbK5b       False  NaN  train
 3       3  6lfxq3CG4xtTiEg7opyCyx       False  NaN  train
 4       4  5vjLSffimiIP26QG5WcN2K       False  NaN  train)